# Deep Neural Scanner — Explore, edit and train

Run the cells in order. `pip install -e ".[public,notebook]"` in the repository first. Public-model cells need internet. The scanner works with inspectable PyTorch models; "neuron" does not mean the same thing across CNNs, RNNs and transformers.

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from neural_scanner import demo_model, scan, set_parameter, train_supervised, save_state


## 1. Scan a dense network

In [ ]:
model = demo_model("dense")
report = scan(model)
print("Total parameters:", report["parameter_count"], "Biases:", report["bias_count"])
pd.DataFrame(report["layers"])

In [ ]:
pd.DataFrame(report["parameters"])

## 2. Edit one neuron weight, inspect before and after

Index `(0,1)` changes the connection from input 1 to dense neuron 0.

In [ ]:
before, after = set_parameter(model, "0.weight", (0, 1), 0.75)
print({"before":before,"after":after})


## 3. Train on synthetic labeled data; check loss

In [ ]:
torch.manual_seed(10)
x = torch.randn(120,4)
y = (x[:,0] + 0.4*x[:,1] > 0).long()
losses = train_supervised(model, x, y, epochs=40, lr=0.02)
sns.lineplot(x=list(range(len(losses))), y=losses)
plt.xlabel("Epoch");plt.ylabel("Training loss");plt.show()

## 4. Scan a CNN and an LSTM

In [ ]:
for kind in ("cnn", "lstm"):
    r=scan(demo_model(kind))
    print(kind, "parameters:",r["parameter_count"], "biases:", r["bias_count"])
    display(pd.DataFrame(r["layers"]))

## 5. Optional: inspect an actual public pretrained CNN (downloads ~45 MB)

In [ ]:
RUN_PUBLIC_CNN = False  # change to True to download
if RUN_PUBLIC_CNN:
    from torchvision.models import resnet18, ResNet18_Weights
    real_cnn = resnet18(weights=ResNet18_Weights.DEFAULT)
    real_report = scan(real_cnn)
    print(real_report["parameter_count"])
    display(pd.DataFrame(real_report["parameters"]).head(25))

## 6. Optional: inspect a small Hugging Face transformer

It is a small test checkpoint, not a general-purpose production model. Inspect only sources you trust. No remote custom code is allowed.

In [ ]:
RUN_PUBLIC_HF = False  # change to True to download
if RUN_PUBLIC_HF:
    from transformers import AutoModelForTokenClassification
    transformer = AutoModelForTokenClassification.from_pretrained("sshleifer/tiny-distilbert-base-cased", trust_remote_code=False)
    transformer_report = scan(transformer)
    print(transformer_report["parameter_count"])
    display(pd.DataFrame(transformer_report["parameters"]).head(30))

## 7. Save a new state version

Saving weights alone is not a complete serialized architecture. Recreate the same architecture to reload.

In [ ]:
# Uncomment to save
# save_state(model, "edited-dense.pt")